In [ ]:
import sys
import os

# Add the src directory to the system path to allow importing custom modules
sys.path.insert(0, os.path.abspath('../src'))

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pandas as pd

# complete machine learning pipeline

* read and prepare data from downloaded csv from Kaggle
* fetch and prepare weather data from open-meteo API
* combine and create features for energy and weather data
* train and test data split - time series split
* train and evaluate model
* GridSearchCV and BayesSearchCV
* save best model 

In [ ]:
# complete train pipeline
from fetch_prepare_data import *
from train_model_predict import *

df_energy_de, start_date, end_date = prepare_energy_data(purpose='modeling')
print(f"Energy data from Kaggle. Shape: {df_energy_de.shape}")

#weather_data_cities = fetch_weather_data_for_cities(in_start_date= start_date, in_end_date = end_date) 
#print(f"Weather data for cities shape: {len(weather_data_cities)}")
#df_weather_germany = merge_weather_data_with_city_weights(weather_data_cities)
#df_weather_germany = create_weather_features(df_weather_germany)

df_weather = prepare_weather_data(in_start_date=start_date, in_end_date=end_date)
print(f"Weather data shape after preparation: {df_weather.shape}")  

#df_energy_weather = combine_energy_weather_dataset(in_energy_df=df_orig, in_weather_df=df_weather)
#print(f"Combined energy and weather data shape: {df_energy_weather.shape}") 



In [ ]:
df_energy_weather = combine_energy_weather_dataset(in_energy_df=df_energy_de, in_weather_df=df_weather)
print(f"Combined energy and weather data shape: {df_energy_weather.shape}") 

In [ ]:
from fetch_prepare_data import *
from train_model_predict import *

df_for_modeling = prepare_data_for_modeling()
print(f"Combined energy and weather data shape: {df_for_modeling.shape}") 

In [ ]:
df_for_modeling.to_pickle('../data/processed/energy_weather_data_for_modeling.pkl')

In [ ]:
df_for_modeling.head()

In [ ]:
# train test split
from lightgbm import LGBMRegressor

#df_for_modeling = pd.read_pickle('../data/processed/energy_weather_data_for_modeling.pkl')

features_train, target_train, features_test, target_test = train_test_split_by_date(df_for_modeling, target_column='EnergyDemand', split_date='2025-01-01')
print(f"Training features shape: {features_train.shape}, Training target shape: {target_train.shape}")
print(f"Testing features shape: {features_test.shape}, Testing target shape: {target_test.shape}")

param_lgbm = { 
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 6, 9]
}   
model_lgbm =LGBMRegressor(random_state=42, force_col_wise=True)


model_pipeline = init_model_pipeline(features_train, 
                                     model_lgbm)
model, best_params = tune_model_bayesian(model_pipeline, 
                                         param_lgbm,
                                         features_train, 
                                         target_train)
print(f"Best hyperparameters: {best_params}")


# prediction

* identify time frame for future feature time frame 
* read and prepare data from SMARD
* fetch and prepare weather data from open-meteo API
* combine and create features for energy and weather data -> future features
* use best model and future features for prediction